# 03. Обучение и оценка моделей деревьев решений

**Цель:** Обучение DecisionTree-моделей с фокусом на online-режим ЛПР.

**Ключевые правила:**
- Канонический источник: `БД-1...2000--2020 (1+2)`
- Основная метрика для ранга: `Macro-F1`
- Основной online-набор: `online_tactical`
- Дефолтная импутация для online-наборов: `-1`

**Вход:** `clean_df_with_rank.csv`

**Выход:**
- Сравнение online-наборов (`online_dispatch`, `online_early`, `online_tactical`)
- Метрики качества (accuracy, f1_macro, f1_weighted)
- Обученные модели и артефакты в `reports/models/`

## 1. Инициализация

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd() / "src"))

from fire_es.model_train import (
    CANONICAL_SOURCE_SHEET,
    FEATURE_SETS,
    TARGET_RANK,
    evaluate_feature_set_cv,
    predict_with_proba,
    prepare_data,
    save_model,
    train_classifier,
    train_regressor,
    visualize_tree,
)

pd.set_option("display.max_columns", 100)
plt.style.use("seaborn-v0_8-whitegrid")

print("Инициализация завершена")

## 2. Загрузка и фильтрация канонического источника

In [ ]:
df = pd.read_csv("clean_df_with_rank.csv")
print(f"Всего записей до фильтрации: {len(df)}")

if "source_sheet" in df.columns:
    df = df[df["source_sheet"] == CANONICAL_SOURCE_SHEET].copy()
    print(f"Фильтрация по source_sheet={CANONICAL_SOURCE_SHEET!r}")

print(f"Записей после фильтрации: {len(df)}")
print("\nРаспределение rank_tz:")
print(df["rank_tz"].value_counts().sort_index())

## 3. Сравнение online feature sets (5-fold CV)

In [ ]:
compare_sets = ["online_dispatch", "online_early", "online_tactical"]
rows = []

for fs in compare_sets:
    result = evaluate_feature_set_cv(
        df=df,
        feature_set=fs,
        target=TARGET_RANK,
        fill_strategy="constant",
        fill_value=-1,
        n_splits=5,
        random_state=42,
    )
    rows.append(
        {
            "feature_set": fs,
            "n_features": result["n_features"],
            "accuracy": result["metrics"]["accuracy_mean"],
            "f1_macro": result["metrics"]["f1_macro_mean"],
            "f1_weighted": result["metrics"]["f1_weighted_mean"],
        }
    )

cv_df = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
cv_df

## 4. Обучение классификатора (online_tactical)

In [ ]:
X, y, feature_names = prepare_data(
    df=df,
    feature_set="online_tactical",
    target=TARGET_RANK,
    fill_strategy="constant",
    fill_value=-1,
)

y_map = {1.0: 1, 1.5: 2, 2.0: 3, 3.0: 4, 4.0: 5, 5.0: 6}
y_int = y.map(y_map).fillna(1).astype(int)

result = train_classifier(
    X,
    y_int,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
)

print("Метрики классификации:")
print(f"  accuracy: {result['metrics']['accuracy']:.4f}")
print(f"  f1_macro: {result['metrics']['f1_macro']:.4f}")
print(f"  f1_weighted: {result['metrics']['f1_weighted']:.4f}")
print(f"  train_size: {result['metrics']['train_size']}")
print(f"  test_size: {result['metrics']['test_size']}")

## 5. Confusion Matrix

In [ ]:
cm = result["metrics"]["confusion_matrix"]
classes = result["metrics"]["classes"]

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## 6. Важность признаков

In [ ]:
feature_importance = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": result["model"].feature_importances_,
    }
).sort_values("importance", ascending=False)

feature_importance

## 7. Визуализация и сохранение модели ранга

In [ ]:
viz_path = visualize_tree(
    result["model"],
    feature_names,
    class_names=[f"Ранг {c}" for c in classes],
    output_path="reports/models/tree_rank_online_tactical.png",
    max_depth=3,
)
print(f"Дерево сохранено: {viz_path}")

saved_rank = save_model(
    model=result["model"],
    metrics=result["metrics"],
    feature_set="online_tactical",
    model_type="rank",
    output_dir="reports/models",
)
print("Модель ранга:", saved_rank["model_path"])
print("Метаданные:", saved_rank["metadata_path"])

## 8. Прогноз с вероятностями (top-3)

In [ ]:
predictions = predict_with_proba(result["model"], X.head(10), top_k=3)
predictions

## 9. Обучение регрессора ресурсов (online_tactical)

In [ ]:
X_res, y_res, feature_names_res = prepare_data(
    df=df,
    feature_set="online_tactical",
    target="equipment_count",
    fill_strategy="constant",
    fill_value=-1,
)

result_res = train_regressor(
    X_res,
    y_res,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
)

print("Метрики регрессии:")
print(f"  mae: {result_res['metrics']['mae']:.4f}")
print(f"  rmse: {result_res['metrics']['rmse']:.4f}")
print(f"  r2: {result_res['metrics']['r2']:.4f}")

saved_res = save_model(
    model=result_res["model"],
    metrics=result_res["metrics"],
    feature_set="online_tactical",
    model_type="resource",
    output_dir="reports/models",
)
print("Модель ресурсов:", saved_res["model_path"])
print("Метаданные:", saved_res["metadata_path"])

## 10. Итог

In [ ]:
print("=" * 60)
print("ИТОГИ")
print("=" * 60)
print(f"Лучшая online-модель ранга: online_tactical")
print(f"accuracy: {result['metrics']['accuracy']:.4f}")
print(f"f1_macro: {result['metrics']['f1_macro']:.4f}")
print(f"f1_weighted: {result['metrics']['f1_weighted']:.4f}")
print()
print("Ресурсы (equipment_count):")
print(f"MAE: {result_res['metrics']['mae']:.4f}")
print(f"RMSE: {result_res['metrics']['rmse']:.4f}")
print(f"R2: {result_res['metrics']['r2']:.4f}")
print("=" * 60)